Do not run the entire notebook at once on a first pass. Some later cells reproduce thesis experiments and may take several minutes or longer. To check that the notebook works, run only Setup, Function definitions, and Quick check first.

# Trust, polarization, and network structure: simulation notebook

This notebook contains the Python simulation code used for the thesis project on trust-based polarization and network structure by Ben Esseling in 2026 for the University of Tilburg, Department of Philosophy. It is a polished version of the original working notebook: the order of the analysis and most code/comments have been kept close to the working version, but the notebook has been cleaned for easier reading and public sharing on GitHub.

Note: AI has been used in the polishing of this version, it was not used to create any of the model or in the notebook that has been used ofr the thesis.

## How to use this notebook

1. Run the setup cell and the function-definition cell first.
2. Run the quick single-simulation example to check that everything works.
3. Run the later experiment sections selectively. Many of the parameter sweeps use hundreds or thousands of runs and may take a long time.

The notebook is intentionally kept as a single file. That makes it easier to upload to GitHub and easier for readers to follow the original research workflow.

## Requirements

The notebook was written for Python 3 and uses the following main packages:

```text
numpy
networkx
matplotlib
pandas
seaborn
ipython
```

If a package is missing, install it in the notebook environment, for example with `pip install numpy networkx matplotlib pandas seaborn ipython`.

## Setup

In [ ]:
# %reset -f  # Optional: uncomment if you want to clear the namespace before running the whole notebook.
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import Image
import sys
import pandas as pd

import seaborn as sns
sns.set_style("whitegrid")

# Optional: uncomment for reproducible random runs while testing.
# np.random.seed(123)

## Function definitions

The following cell contains the model functions from the working notebook, including graph generation, updating rules, simulations, and plotting helpers.

In [ ]:
def generate_graph(a, i, con, tics):    
    graph = np.zeros((a+i,a+i))
    arr_credence = np.zeros((tics,a+i))
    exp_results = np.zeros((tics,a+i))
    
    #randomly fill up the graph with set connectivity for all agents, and give all influencers full connections. No-one is connected to themselves.
    
    for j1 in range(a):
        for j2 in range(j1):
            graph[j1,j2] = np.random.binomial(1,con);
    
    for j1 in range(a,a+i):
        for j2 in range(j1):
            graph[j1,j2] = 1;
    
    for j1 in range(a+i):
        for j2 in range(j1):
            graph[j2,j1] = graph[j1,j2];
    return graph, arr_credence, exp_results



def generate_graph_smallworld(a, i, tics, k = 8, p = 0.1):
    graph = np.zeros((a+i,a+i))
    arr_credence = np.zeros((tics,a+i))
    exp_results = np.zeros((tics,a+i))

    for j1 in range(a,a+i):
        for j2 in range(j1):
            graph[j1,j2] = 1;

    for j1 in range(a+i):
        for j2 in range(j1):
            graph[j2,j1] = graph[j1,j2];
    
    G = nx.connected_watts_strogatz_graph(a, k, p);
    A = nx.to_numpy_array(G, dtype=int)

    graph[:a,:a] = A

    return graph, arr_credence, exp_results

    


def generate_graph_inoutgroup(a, i, num_groups, con_in, con_out, tics):
    graph = np.zeros((a+i, a+i))
    arr_credence = np.zeros((tics, a+i))
    exp_results = np.zeros((tics, a+i))

    #fill the graph such that you have num_groups of groups that are internally connected with con = con_in, and externally (agents in different groups) with con = con_out

    base = a // num_groups
    remainder = a % num_groups
    sizes = [base + 1 if a_div < remainder else base for a_div in range(num_groups)]

    #assign each of the a non-influencer agents to a group id
    group_id = np.empty(a, dtype=int)
    idx = 0
    for g, sz in enumerate(sizes):
        group_id[idx:idx + sz] = g
        idx += sz
    #print(group_id)
    
    #fill connections among non-influencers (agents 0..a-1), only fill lower triangle and then mirror
    for j1 in range(a):
        for j2 in range(j1):
            p = con_in if group_id[j1] == group_id[j2] else con_out
            graph[j1, j2] = np.random.binomial(1, p)

    #influencers
    for j1 in range(a, a+i):
        for j2 in range(j1):
            graph[j1, j2] = 1

    #mirror
    for j1 in range(a+i):
        for j2 in range(j1):
            graph[j2,j1] = graph[j1,j2];

    #print(graph)

    return graph, arr_credence, exp_results



def build_graph(a, i, tics, topology):
    if topology["type"] == "random":
        con = topology["con"]
        return generate_graph(a, i, con, tics)

    elif topology["type"] == "groups":
        return generate_graph_inoutgroup(a, i, topology["num_groups"], topology["con_in"], topology["con_out"], tics)

    elif topology["type"] == "smallworld":
        return generate_graph_smallworld(a, i, tics, topology["k"], topology["p"])

    else:
        raise ValueError(f"Unknown topology {topology['type']}")




def initial_credences(arr_credence):
    for agent in range(len(arr_credence[0,:])): #apply fully initial credences to all agents
        arr_credence[0,agent] = np.random.rand()
    return arr_credence



def hist_of_connections(graph, a, i):
    one_count = np.sum(graph == 1, axis=1); #number of ones per row
    unconnected_agents = np.sum(one_count == i); #an agent is unconnected if it has only i connections
    #print(unconnected_agents)
    plt.hist(one_count,bins=range(a+i+1));



def plot_graph(graph, size = 4):
    G = nx.from_numpy_array(graph)
    pos = nx.spring_layout(G)
    plt.figure()
    nx.draw(G, pos, with_labels=True)
    fig = plt.gcf()

    # Get current size
    w, h = fig.get_size_inches()
    aspect_ratio = h / w

    # Set new width (in inches)
    new_width = size
    new_height = new_width * aspect_ratio

    fig.set_size_inches(new_width, new_height)
    plt.show()



def agent_experiment(credence, epsilon, n):
    if credence > 0.5:
        k = np.random.binomial(n, 0.5 + epsilon)
        exp = 1 # 1 if agent experiments, 0 if not
        updated_credence = bayesian_update_self(k, credence, n, epsilon)
    else:
        k = -1
        exp = 0
        updated_credence = credence
    return k, updated_credence, exp



def bayesian_update_self(k, credence, n, epsilon):
    updated_credence = 1 / (1 + (1 - credence) * (((0.5 - epsilon) / (0.5 + epsilon)) ** (2 * k - n)) / credence) #formula from jweisber
    return updated_credence #closed-form Bayesian update for a binomial model with two competing hypotheses, here the two competing hypotheses are p=0.5+epsilon and p=0.5-epsilon, as we assume that the unknown option is either of the two.



def update_on_neighbours(own_credence, neighbour_credence, k, n, epsilon, m, disc_func = 2):
    #as we have new, non-overlapping, evidence for all agents every tic, multiplying likelihood ratios one-by-one is the same as multiplying them all together, so we don't have to take this into acount when updating credence.
    #the formulas below are acquired from jwesiber's ABM
    p_E_H = (0.5 + epsilon)**k * (0.5 - epsilon)**(n - k)         # P(E|H)  = p^k (1-p)^(n-k)   # there should be an nCk here but it cancels out in the fourth line
    p_E_nH = (0.5 - epsilon)**k * (0.5 + epsilon)**(n - k)        # P(E|~H) = (1-p)^k p^(n-k)   # there should be an nCk here but it cancels out in the fourth line
    p_E    = own_credence * p_E_H + (1 - own_credence) * p_E_nH   # P(E) = P(E|H) P(H) + P(E|~H) P(~H)

    p_H_E  = own_credence * p_E_H / p_E                           # P(H|E)  = P(H) P(E|H)  / P(E)
    p_H_nE = own_credence * (1 - p_E_H) / (1 - p_E)               # P(H|~E) = P(H) P(~E|H) / P(~E)

    if disc_func == 1:
        q_E = max(1 - abs(own_credence - neighbour_credence) * m * (1 - p_E), 0)  # O&W's Eq. 1 (anti-updating)
    elif disc_func == 2:
        q_E = 1 - min(1, abs(own_credence - neighbour_credence) * m) * (1 - p_E)    # O&W's Eq. 2, standard use this one
    else:
        raise ValueError(f"disc_func must be 1 or 2, got {disc_func}")   #making sure i parse 1 or 2, not 0 or 1

    own_credence = p_H_E * q_E + p_H_nE * (1 - q_E)               # Jeffrey's Rule: P'(H) = P(H|E) P'(E) + P(H|~E) P'(~E)
    return own_credence



def simulation1(tics, a, i, n, con, epsilon, m, output_text=True, disc_func=2): #output_text is True or False, dependant if you want tou output the outcome message
       # per tic
       # 1. experiment for all agents at same time (synchronous):
       #    if credence > .5:
       #          draw k and immediately bayesian update self
       #    else: no experiment, no change
       # 2. social updating all agents at same time, so not order dependant:
       #    take snapshot of post-experiment credences
       #    each agent first calculates Jeffrey update based on neighbours who experimented but dont update yet, so others update based on your old credence.
       #    then all agents update their credence at once
       # 3. Stopping rule:.
       #        outcome 1: all > .99
       #        outcome 2: all ≤ .5
       #        outcome 3: m * d > 1

    outcome_parsed = 0
    graph, arr_credence, exp_results = generate_graph(a, i, con, tics)
    arr_credence = initial_credences(arr_credence)
    
    for t in range(1,tics):
        for agent in range(len(arr_credence[0, :])):                #experiment (doesnt experiment if credence < 0.5)
            credence = arr_credence[t-1, agent]                     #get credence from last tic, this is why we have tics - 1 amount of updating
            k, updated_credence, exp = agent_experiment(credence, epsilon, n)
            exp_results[t,agent] = k                                #logging the experiment results
            arr_credence[t, agent] = updated_credence               #here I make the step to this tic's time, which is when i Bayesian update on own experiment or keep it the same
        
        post_exp_credence = arr_credence[t,:].copy()                #credences of all agents after inquiry of tic t, before social updating, done so that neigbours update on not yet-socially updated credences, otherwise order would matter
        post_social_credence = post_exp_credence.copy()                      #working array of credences of all agents post-social updating, at the end of tic t, this will become arr_credences[t,:]
        
        for agent in range(a+i):                                    #for all agents, we are going to update based on neighbours
            for neighbour in range(a+i):                            #going over all other agents
                if graph[agent, neighbour] == 1 and exp_results[t, neighbour] != -1:           #if other agent is neighbour (note you are not neighbour to yourself) and neighbour did experiment
                    post_social_credence[agent] = update_on_neighbours(post_social_credence[agent], post_exp_credence[neighbour], exp_results[t,neighbour], n, epsilon, m, disc_func)   #symmetric updating i.e. you update on not yet socially-updated neighbours

        arr_credence[t,:] = post_social_credence
        
        #note to self: on tic 0 the initial credences are set in arr_credence[0,:], then every tic you take the credence from last tic as start.
        #this means that the evidence in exp_results[j,:] is the evidence that formed the credences in arr_credence[j,:].
        #convergence checks (reference-style; no tol/lag)
        cred_now = post_social_credence

        if np.all(cred_now > 0.99):
            outcome = 1
        elif np.all(cred_now <= 0.5):
            outcome = 2
        elif polarized_ow(cred_now, m):
            outcome = 3
        else:
            outcome = 0

        t_end = t
        if outcome != 0:
            if output_text:
                print("Stopped with outcome", outcome, "at tic", t_end)
            break

    if outcome == 0 and output_text:
        print("No convergence achieved, outcome 0, finished at tic ", t_end)

    return arr_credence, exp_results, t_end, outcome




def simulation2(tics, a, i, n, con, epsilon, m, output_text=False, disc_func=2, topology = None):
    # per tic
    # 1. everyone experiments; they set k but DONT update Bayesian
    # 2. social updating is done order dependant:
    #    for each agent (in increasing index order):
    #            a) If the agent experimented, Bayes-update on own evidence first
    #            b) Then Jeffrey-update on each neighbor's evidence (using neighbor's current credence).
    # Stopping condition:
    #   - stop if all credences > .99  (outcome 1)
    #   - stop if all credences <= .5  (outcome 2)
    #   - stop if d * m > 1 meaning polarized state
    #   - else continue until tics cap (outcome 0)

    if topology is None:
        topology = {"type": "random", "con": con}
    
    graph, arr_credence, exp_results = build_graph(a, i, tics, topology)

    
    arr_credence = initial_credences(arr_credence)

    outcome = 0
    t_end = 0

    for t in range(1, tics):
        #start this tic from last tic credences where no-one has updated yet
        arr_credence[t, :] = arr_credence[t - 1, :]

        #experiments: ONLY draw k (store evidence). No Bayesian update here.
        for agent in range(a+i):
            cred = arr_credence[t - 1, agent]
            if cred > 0.5:
                exp_results[t, agent] = np.random.binomial(n, 0.5 + epsilon)
            else:
                exp_results[t, agent] = -1

        #social updating in order, so dependant on the order we do it
        for agent in range(a+i):
            #self Bayes update first (if experimented)
            if exp_results[t, agent] != -1:
                arr_credence[t, agent] = bayesian_update_self(exp_results[t, agent], arr_credence[t, agent], n, epsilon)

            # jeffrey updating, excluding self
            for neighbour in range(a+i):
                if neighbour == agent:
                    continue
                if graph[agent, neighbour] == 1 and exp_results[t, neighbour] != -1:
                    arr_credence[t, agent] = update_on_neighbours(arr_credence[t, agent], arr_credence[t, neighbour], exp_results[t, neighbour], n, epsilon, m, disc_func=disc_func)

        #convergence checks (reference-style; no tol/lag)
        cred_now = arr_credence[t, :]

        if np.all(cred_now > 0.99):
            outcome = 1
        elif np.all(cred_now <= 0.5):
            outcome = 2
        elif polarized_ow(cred_now, m):
            outcome = 3
        else:
            outcome = 0

        t_end = t
        if outcome != 0:
            if output_text:
                print("Stopped with outcome", outcome, "at tic", t_end)
            break

    if outcome == 0 and output_text:
        print("No convergence achieved, outcome 0, finished at tic ", t_end)

    return arr_credence, exp_results, t_end, outcome



def line_plot_credence(arr_credence, exp_results, t, size = 4, plot_tmax = False):   #plots the credences over time in red lines
  
    for agent in range(len(arr_credence[0,:])):
        plt.plot(range(len(arr_credence[:,0])), arr_credence[:, agent], color="red", linewidth=0.5, alpha=0.6)
    
    plt.xlabel("Time (tic)")
    plt.ylabel("Credence")
    plt.title("Credence Evolution Over Time")
    plt.ylim(-0.05, 1.05)
    if plot_tmax:
        plt.xlim(0,len(arr_credence[:,0]))
    else:
        plt.xlim(0, t)
    
    fig = plt.gcf()

    # Get current size
    w, h = fig.get_size_inches()
    aspect_ratio = h / w

    # Set new width (in inches)
    new_width = size
    new_height = new_width * aspect_ratio

    fig.set_size_inches(new_width, new_height)
    
    plt.show()




def polarized_ow(credence, m, low=0.5, high=0.99):
    left = credence < low
    right = credence > high

    #must be only in the two regions, and both sides must exist
    if not np.all(left | right):
        return False
    if not (np.any(left) and np.any(right)):
        return False

    min_believer = np.min(credence[right])     # weakest believer
    max_disbeliever = np.max(credence[left])   # strongest disbeliever
    d = min_believer - max_disbeliever

    return (m * d) >= 1



def run_simulation(tics, a, i, n, con, epsilon, m, output_text=True, type_sim=2, disc_func=2): #2 is O'Connor and Weatherall, and jweisber's implimentation
    if type_sim == 1:
        arr_credence, exp_results, t, outcome = simulation1(tics, a, i, n, con, epsilon, m, output_text, disc_func)
    if type_sim == 2:
        arr_credence, exp_results, t, outcome = simulation2(tics, a, i, n, con, epsilon, m, output_text, disc_func)
    return arr_credence, exp_results, t, outcome

## Quick check: one simulation

Run this cell after the setup and function definitions to confirm that the notebook works. It creates a single simulation plot.

### One single credence trajecotry plot. 
Here I create the plot of a single simulation untill convergence or untill the max tics

In [ ]:
arr = run_simulation(1001, 50, 0, 10, 1, 0.01, 1.5, 2, 2) #tics, a, i, n, con, epsilon, m, version=2, discounting_function=2
line_plot_credence(*arr[:3])

## Thesis experiments and exploratory analysis

The remaining cells reproduce and explore the simulations used in the thesis. They are kept close to the working notebook, including original comments and exploratory notes. Some cells are long-running; reduce `runs` before running them if you only want a quick test (e.g. runs = 10 should aready give some idea of the behaviour of most simulations).

### I will innitially try to replicate O'connor and Weatherall, and jweisbers', results:
My initial problem is that I get too high convergence for lower m when comapring to O'Connor and Waetherall, and jweisbers. This is either the code that does not do the same thing, or the parameters that are set wrong.

It turned out that when I originally coded the simulation function and the order agents update their beliefs, I assumed it would be symmetric i.e. not order dependant; that agents would first calculate what their own updated credence would be but still gave their original credence to their neighbours, this would make sense because then it doesnt matter in what order you update (This is simulation1 fucntion). Instead both O&W and jweisbers do it asymmetrically, meaning agents immediately update their credence based on their neighbours, and then others use this updated credence in the same tic (This is simulation 2 function). This leads to lower polarization for lower m.

Important to note here thus is that this model is highly dependant on not only parameters, as O&W already discuss, but also simply model setup. Changing the order in which agents update leads to a higher proportion of polarized runs, without changing anything else. The general behaviour is still the same but the specifics differ quite a lot. To take from this is that even if the specifics might look nice (e.g. it rounds off at m = 2), we can only ever look at the abstracted behaviour (e.g. that it converges to polarized runs).

### Below is a reproduction of fig 2 from O&W but with simtype 1 (synchronous updating)

In [ ]:
ms = [1, 1.1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3]
runs = 100 #runs per m
agent_sizes = [6, 10, 20]
arr_runs = np.zeros((len(agent_sizes), len(ms), runs, 2))
total = len(agent_sizes) * len(ms) * runs
done = 0
for a in range(len(agent_sizes)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...,
            _, _, t, output = run_simulation(10001, agent_sizes[a], 0, 50, 1, 0.2, ms[m], False, 1, 2); #tics, a, i, n, con, epsilon, m, output_text, version, discounting_function
            arr_runs[a, m, r, 0] = t
            arr_runs[a, m, r, 1] = output
            
            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

            
polarized_prop = np.zeros((len(agent_sizes), len(ms)))
for a in range(len(agent_sizes)):
    for m in range(len(ms)):
        outputs = arr_runs[a, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[a, m] = polarized / runs
        
plt.figure()
for a_idx, a in enumerate(agent_sizes):
    plt.plot(ms, polarized_prop[a_idx, :], marker="o", label=f"a = {a}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("figure2_replication_simtype1.pdf", bbox_inches="tight")
plt.show()

### Below is the reproduction of figure 2 from O&W 2018, and jweisbers blog (https://jonathanweisberg.org/post/ow/)
Note, runs = 100 takes roughly 20 seconds, but atleast 500 is required for a strong statistical plot. Running takes a long time. The "figure2_replication.pdf" plot saved is a 1000 runs plot, with the following parameters: run_simulation(10001, agent_sizes[a], 0, 50, 1, 0.2, ms[m], False). It took ~3 minutes.

In [ ]:
ms = [1, 1.1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3]
runs = 50 #runs per m
agent_sizes = [6, 10, 20]
arr_runs = np.zeros((len(agent_sizes), len(ms), runs, 2))
for a in range(len(agent_sizes)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = run_simulation(10001, agent_sizes[a], 0, 50, 1, 0.2, ms[m], False, 2, 2); #tics, a, i, n, con, epsilon, m, output_text, version, discounting_function
            arr_runs[a, m, r, 0] = t
            arr_runs[a, m, r, 1] = output
            
polarized_prop = np.zeros((len(agent_sizes), len(ms)))
for a in range(len(agent_sizes)):
    for m in range(len(ms)):
        outputs = arr_runs[a, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[a, m] = polarized / runs
        
plt.figure()
for a_idx, a in enumerate(agent_sizes):
    plt.plot(ms, polarized_prop[a_idx, :], marker="o", label=f"a = {a}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("figure2_replication_final_v2.pdf", bbox_inches="tight")
plt.show()

### Next I want to replicate figure 6 from O&W to make sure I'm using the right model

Now that figure 2 is replicated and I know i have the right macro behaviour (the right proportion of runs polarize), I want to check if indeed my agents also behave the same as O&W on micro scale (whether the right proportion of agents hold a true or false belief). 

I seem to only be able to get a curve that corresponds to using the no anti-updating rule: while using both discount function their output is exactly the same. This means I must not be getting in the area of distance between agents where d*m > 1, namely where these two formulas start to change. That makes sense because when we get in that area, we see that the polarized_ow function, which decides when it is polarized according to O&W, stops the simulation. So if I want to get the same results as O&W for the anti-updating function, I will have to let the simulation keep running beyond its polarizing threshold. I won't do this as this is outside the scope of what I want to achieve, and I have seen that the behaviour for the anti-updating function is exactly as expected.

the saved plot is made for ms = [1.1, 1.5, 2, 2.5], runs = 1000, tics = 50001, a = 20, n = 10, con = 1, epsilon = 0.2

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 50 #runs per m
formulas = [2]
arr_runs = np.zeros((len(formulas), len(ms), runs, 2))

false_prop = np.zeros((len(formulas), len(ms)))

for form in range(len(formulas)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...,
            arr_credence, _, t, _ = run_simulation(50001, 20, 0, 10, 1, 0.2, ms[m], False, 2, formulas[form]); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[form, m] = false_beliefs / (false_beliefs + true_beliefs)

        
plt.figure()
labels = {1: "Anti-updating", 2: "No anti-updating"}

for form_idx, form in enumerate(formulas):
    plt.plot(ms, false_prop[form_idx, :], marker="o", label=labels[form])

plt.xlabel("m")
plt.ylabel("Percentage of False Beliefs")
plt.ylim(0, 0.6)

plt.legend()
#plt.savefig("figure6_replication.pdf", bbox_inches="tight")
plt.show()

Below is the same reproduction but now for simtype 1 (synchronous updating)

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 50 #runs per m
formulas = [2]
arr_runs = np.zeros((len(formulas), len(ms), runs, 2))

false_prop = np.zeros((len(formulas), len(ms)))

for form in range(len(formulas)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            arr_credence, _, t, _ = run_simulation(50001, 20, 0, 10, 1, 0.2, ms[m], False, 1, formulas[form]); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[form, m] = false_beliefs / (false_beliefs + true_beliefs)

        
plt.figure()
labels = {1: "Anti-updating", 2: "No anti-updating"}

for form_idx, form in enumerate(formulas):
    plt.plot(ms, false_prop[form_idx, :], marker="o", label=labels[form])

plt.xlabel("m")
plt.ylabel("Percentage of False Beliefs")
plt.ylim(0, 0.6)

plt.legend()
#plt.savefig("figure6_replication_simtype1.pdf", bbox_inches="tight")
plt.show()

### Having verified the model, next set up different types of networks e.g. con < 1

First, lets see what different connectivity does does.

Having both low c and low m lead to cases where no convergence is found within the max tics (of eg 10000), this is due to low c and low m runs having very slow convergence with a lot of low or non-connected agents. so counter-intuitive is that higher amount of agents will actually speed the simulation up -> convergence is reached earlier. Saved plot is made at tics = 10001, agents = 100, epsilon = 0.2, n = 50, ms = [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.75, 2, 2.25, 2.5], runs = 1000, cons = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]. 
Running this took ~80 minutes. Image is connectivity_over_m_final.

In [ ]:
agents = 100
epsilon = 0.2
n = 50
ms = [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.75, 2, 2.25, 2.5]
runs = 10 #runs per m
cons = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]

arr_runs = np.zeros((len(cons), len(ms), runs, 2))

for c in range(len(cons)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = run_simulation(10001, agents, 0, n, cons[c], epsilon, ms[m], False, 2, 2); #tics, a, i, n, con, epsilon, m, output_text, version, discounting_function
            arr_runs[c, m, r, 0] = t
            arr_runs[c, m, r, 1] = output
        print(cons[c],ms[m])
            
polarized_prop = np.zeros((len(cons), len(ms)))
for c in range(len(cons)):
    for m in range(len(ms)):
        outputs = arr_runs[c, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[c, m] = polarized / runs
        
plt.figure()
for c_idx, c in enumerate(cons):
    plt.plot(ms, polarized_prop[c_idx, :], marker="o", label=f"connectivity = {c}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("connectivity_over_m.pdf", bbox_inches="tight")
plt.show()

Clearly we see that higher connectivity leads to a stronger tendency to become polarized, what happens to the degree of true and false beliefs of the agents for different connectivities? Are they different than in the case for full connectivity? i.e. does connectivity change the fraction of true or false beliefs as well?

First I want to check how these proportions of beliefs change for different amounts of agents over m, so roughly what was done in the reproduction of fig 6 from O&W. But also how this behaves over n and epsilon, just to get an idea on how everything moves.

NOTE that an agent achieves either a true or false belief at the end, nothing else. Hence only true beliefs are plotted, as false beliefs show the same curve in reverse.

(this plot was made with the following parameters: ms = [1.1, 1.5, 2, 2.5], runs = 500, agents = [6, 20, 100], ticks = 50001, n = 2 and 50, con = 1, epsilon = 0.2.)

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 50 #runs per m
agents = [6, 20, 100]
arr_runs = np.zeros((len(agents), len(ms), runs, 2))

false_prop = np.zeros((len(agents), len(ms)))
true_prop = np.zeros((len(agents), len(ms)))

for a in range(len(agents)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            arr_credence, _, t, _ = run_simulation(50001, agents[a], 0, 2, 1, 0.2, ms[m], False, 2); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[a, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[a, m] = true_beliefs / (false_beliefs + true_beliefs)



plt.figure()
plt.plot(ms, true_prop[0, :],
         marker="o",
         linestyle="-",
         color="tab:blue",
         label="6 agents, n=2")

plt.plot(ms, true_prop[1, :],
         marker="o",
         linestyle="-",
         color="tab:orange",
         label="20 agents, n=2")

plt.plot(ms, true_prop[2, :],
         marker="o",
         linestyle="-",
         color="tab:green",
         label="100 agents, n=2")

        
arr_runs = np.zeros((len(agents), len(ms), runs, 2))
false_prop = np.zeros((len(agents), len(ms)))
true_prop = np.zeros((len(agents), len(ms)))

for a in range(len(agents)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            arr_credence, _, t, _ = run_simulation(50001, agents[a], 0, 50, 1, 0.2, ms[m], False, 2); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[a, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[a, m] = true_beliefs / (false_beliefs + true_beliefs)
        

plt.plot(ms, true_prop[0, :],
         marker="s",
         linestyle="--",
         color="tab:blue",
         label="6 agents, n=50")

plt.plot(ms, true_prop[1, :],
         marker="s",
         linestyle="--",
         color="tab:orange",
         label="20 agents, n=50")

plt.plot(ms, true_prop[2, :],
         marker="s",
         linestyle="--",
         color="tab:green",
         label="100 agents, n=50")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [5, 4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("beliefs_over_agents_final.pdf", bbox_inches="tight")
plt.show()

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 500 #runs per m
agent_sizes = [6, 20, 100]
arr_runs = np.zeros((len(agent_sizes), len(ms), runs, 2))
for a in range(len(agent_sizes)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = run_simulation(10001, agent_sizes[a], 0, 50, 1, 0.2, ms[m], False, 2, 2); #tics, a, i, n, con, epsilon, m, output_text, version, discounting_function
            arr_runs[a, m, r, 0] = output

            
polarized_prop = np.zeros((len(agent_sizes), len(ms), 2))
for a in range(len(agent_sizes)):
    for m in range(len(ms)):
        outputs = arr_runs[a, m, :, 0]
        polarized = np.sum(outputs == 3)
        polarized_prop[a, m, 0] = polarized / runs
        
plt.figure()
plt.plot(ms, polarized_prop[0, :, 0],
         marker="o",
         linestyle="-",
         color="tab:blue",
         label="6 agents, n=2")

plt.plot(ms, polarized_prop[1, :, 0],
         marker="o",
         linestyle="-",
         color="tab:orange",
         label="20 agents, n=2")

plt.plot(ms, polarized_prop[2, :, 0],
         marker="o",
         linestyle="-",
         color="tab:green",
         label="100 agents, n=2")


for a in range(len(agent_sizes)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = run_simulation(10001, agent_sizes[a], 0, 2, 1, 0.2, ms[m], False, 2, 2); #tics, a, i, n, con, epsilon, m, output_text, version, discounting_function
            arr_runs[a, m, r, 1] = output

        
for a in range(len(agent_sizes)):
    for m in range(len(ms)):
        outputs = arr_runs[a, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[a, m, 1] = polarized / runs

plt.plot(ms, polarized_prop[0, :, 1],
         marker="s",
         linestyle="--",
         color="tab:blue",
         label="6 agents, n=50")

plt.plot(ms, polarized_prop[1, :, 1],
         marker="s",
         linestyle="--",
         color="tab:orange",
         label="20 agents, n=50")

plt.plot(ms, polarized_prop[2, :, 1],
         marker="s",
         linestyle="--",
         color="tab:green",
         label="100 agents, n=50")

plt.xlabel("m")
plt.ylabel("Proportion of polarized runs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [5, 4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("polarization_over_agents_final.pdf", bbox_inches="tight")
plt.show()

So we see that when agents experiment more, they achieve better beliefs, which is expected and confirms a sanity check. What we also see is that for the same n, a higher number of agents lead to a higher proportion of them having true beliefs, which can be explained by the bigger total pull that agents >0.5 credence have on the agents below them (i think).

We expect to see the same for epsilon: higher epsilon means clearer experiments, so more true beliefs.

(these plots were made with the following parameters: ms = [1.1, 1.5, 2, 2.5], runs = 500, agents = [6, 20, 100], ticks = 50001, n = 50, con = 1, epsilon = 0.05 and 0.25.)

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 500 #runs per m
agents = [6, 20, 100]
arr_runs = np.zeros((len(agents), len(ms), runs, 2))

false_prop = np.zeros((len(agents), len(ms)))
true_prop = np.zeros((len(agents), len(ms)))

for a in range(len(agents)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...,
            arr_credence, _, t, _ = run_simulation(50001, agents[a], 0, 50, 1, 0.05, ms[m], False, 2); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[a, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[a, m] = true_beliefs / (false_beliefs + true_beliefs)

plt.figure()
plt.plot(ms, true_prop[0, :],
         marker="o",
         linestyle="-",
         color="tab:blue",
         label="True beliefs, 6 agents (epsilon = 0.05)")

plt.plot(ms, true_prop[1, :],
         marker="o",
         linestyle="-",
         color="tab:orange",
         label="True beliefs, 20 agents (epsilon = 0.05)")

plt.plot(ms, true_prop[2, :],
         marker="o",
         linestyle="-",
         color="tab:green",
         label="True beliefs, 100 agents (epsilon = 0.05)")


arr_runs = np.zeros((len(agents), len(ms), runs, 2))
false_prop = np.zeros((len(agents), len(ms)))
true_prop = np.zeros((len(agents), len(ms)))

for a in range(len(agents)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            arr_credence, _, t, _ = run_simulation(50001, agents[a], 0, 50, 1, 0.25, ms[m], False, 2); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[a, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[a, m] = true_beliefs / (false_beliefs + true_beliefs)

        
plt.plot(ms, true_prop[0, :],
         marker="s",
         linestyle="--",
         color="tab:blue",
         label="True beliefs, 6 agents (epsilon = 0.25)")

plt.plot(ms, true_prop[1, :],
         marker="s",
         linestyle="--",
         color="tab:orange",
         label="True beliefs, 20 agents (epsilon = 0.25)")

plt.plot(ms, true_prop[2, :],
         marker="s",
         linestyle="--",
         color="tab:green",
         label="True beliefs, 100 agents (epsilon = 0.25)")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [5, 4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("beliefs_over_agents_epsilon.pdf", bbox_inches="tight")
plt.show()

We see an extremely similar result which confirms the suspicion that epsilon and n serve the same purpose; higher values correspond to providing clear results in the experimentation that lead to more true beliefs. 

Furthermore we see clearly that increasing the number of agents leads to a higher fraction of true believers. Combining this with the result from earlier that more agents lead to a higher proportion of polarized runs can be explained because having more agents yield a larger group of very low agents that are not influenced by many others, yet they are pulled harder by the above group if they are indeed changing. (WILL NEED TO PLOT SOME LINE PLOTS FOR THIS).


Let's continue now with what happens to the above plots, so true and false beliefs when changing the connectivity.

(the following plot uses ms = [1.1, 1.5, 2, 2.5], runs = 100, agents = 50, ticks = 50001, n = 50, con = [0.1, 0.25, 0.5, 0.75, 1], epsilon = 0.2. Takes ~20 minutes)

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 100 #runs per m
cons = [0.1, 0.25, 0.5, 0.75, 1]
arr_runs = np.zeros((len(cons), len(ms), runs, 2))

false_prop = np.zeros((len(cons), len(ms)))
true_prop = np.zeros((len(cons), len(ms)))

for c in range(len(cons)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            arr_credence, _, t, _ = run_simulation(50001, 50, 0, 50, cons[c], 0.2, ms[m], False, 2); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[c, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[c, m] = true_beliefs / (false_beliefs + true_beliefs)

plt.figure()
for c_idx, c in enumerate(cons):
    plt.plot(ms, true_prop[c_idx, :], marker="o", label=f"connectivity = {c}")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("beliefs_over_cons.pdf", bbox_inches="tight")
plt.show()

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 50 #runs per m
cons = [0.1, 0.25, 0.5, 0.75, 1]
arr_runs = np.zeros((len(cons), len(ms), runs, 2))

false_prop = np.zeros((len(cons), len(ms)))
true_prop = np.zeros((len(cons), len(ms)))

for c in range(len(cons)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=10, con=1, epsilon=0.01, m=...
            arr_credence, _, t, _ = run_simulation(50001, 50, 0, 10, cons[c], 0.01, ms[m], False, 2); #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[c, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[c, m] = true_beliefs / (false_beliefs + true_beliefs)

plt.figure()
for c_idx, c in enumerate(cons):
    plt.plot(ms, true_prop[c_idx, :], marker="o", label=f"connectivity = {c}")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("beliefs_over_cons_50a_n10_e001.pdf", bbox_inches="tight")
plt.show()

So we see that higher connectivity increases both polarization and the proportion of true beliefs. Although for the low-information case, it seems that this does not have as much of an effect anymore How can we explain this?

First let's look at single simulations and the trajectories of the credences. Start with the effect that m has, we see that with incresing m, there is less clear convergence (mostly to a true belief), as we see now that the credence trajectories become more and more ruggid and that more initial low credence agents are left behind. (WHAT DOES THIS MEAN?).

In [ ]:
arr = simulation2(1001, 50, 0, 10, 1, 0.01, 2.5, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 50, n = 10, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

# arr = simulation2(1001, 50, 0, 10, 1, 0.01, 1.5, output_text=True, disc_func=2)
# #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
# print("agents = 50, n = 10, epsilon = 0.01, m = 1.5")
# line_plot_credence(*arr[:3])

# arr = simulation2(1001, 50, 0, 10, 1, 0.01, 2, output_text=True, disc_func=2)
# #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
# print("agents = 50, n = 10, epsilon = 0.01, m = 2")
# line_plot_credence(*arr[:3])

# arr = simulation2(1001, 50, 0, 10, 1, 0.01, 1.4, output_text=True, disc_func=2)
# #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
# print("agents = 50, n = 10, epsilon = 0.01, m = 1.4")
# line_plot_credence(*arr[:3])

# arr = simulation2(1001, 50, 0, 10, 1, 0.01, 1.5, output_text=True, disc_func=2)
# #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
# print("agents = 50, n = 10, epsilon = 0.01, m = 1.5")
# line_plot_credence(*arr[:3])

Furthermore it is very important to note the effect that n and epsilon have on the credence trajectories. More certain information leads to an almost immediate convergence of a majority of the agents. This shows that if we are not operating in the parameter range where n in (1,20) and  epsilon in (0.01, 0.05), we expect to see very simple convergence to a polarized state where some agents get left behind seamingly randomly.

A sidenote is that I find tere to be little difference between lineplots of simulation type 1 (with symmetric social updating) vs simulation type 2 (with asymmetric updating). This might be nice to investigate further later but not now.

In [ ]:
arr = simulation2(1001, 100, 0, 100, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 100, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 50, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 50, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 20, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m,, output_text, version, disc_func
print("agents = 100, n = 20, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 10, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 5, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 5, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 1, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 1, epsilon = 0.01, m = 1.1")
line_plot_credence(*arr[:3])

And now, for the effect that connectivity has on these lineplots, lets look at a = 100, m = 1.1 (interesting range from connectivity over m), n = 10 epsilon = 0.01 which is where i expect to see the most effect. We see that for lower connectivity we have significantly slower convergence (see the x axis) than higher connectivity, which is expected as more connectivity means more pull on everyone. Higher connectivity has also shown to lead to a higher proportion of true belief in agents, but also to a higher proportion of polarization.

In [ ]:
arr = simulation2(1001, 100, 0, 10, 0.1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, con = 0.1")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 10, 0.25, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, con = 0.25")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 10, 0.5, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, con = 0.5")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 10, 0.75, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, con = 0.75")
line_plot_credence(*arr[:3])

arr = simulation2(1001, 100, 0, 10, 1, 0.01, 1.1, output_text=True, disc_func=2)
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, con = 1")
line_plot_credence(*arr[:3])

### Now I wish to proceed to implementing different graph structures
Lets start with a simple case where we have a set number of groups of equal size, and the agents have a probability of connecting with another agent in the same group of con_in, and a probability of connecting with a different agent in a different group of con_out. Logically it's interesting to look at the case con_in >> con_out such that we have clustering:

In [ ]:
graph, _, _ = generate_graph_inoutgroup(50, 0, 3, 0.9, 0.05, 1) #a, i, num_groups, con_in, con_out, tics
plot_graph(graph)

Now that we have a simple in-out group graph, lets play around with it, how do our lineplots look like for 2 or 3 groups?

In [ ]:
arr = simulation2(10001, 100, 0, 10, 1, 0.01, 1.1, output_text=True, disc_func=2, topology={"type": "groups", "num_groups": 2, "con_in": 1, "con_out": 0.05})
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, groups = 2, con_in = 1, con_out = 0.05")
line_plot_credence(*arr[:3])

What effect does the number of groups have on the proportion of polarized results? i expect a very high amount of influence as now, with low con_out, you would expect every group to mostly behave on their own, meaning that if one group is polarized, information from a second group first passes by a limited amout of actors, and only the influence it has on them will possibly influence the polarized members of that group. I wil simulate with 20 agents, n = 50 and epsilon = 0.2 to refind the results for figure 2 of O&W in our group = 1 case. Note that we have to put con_out = 0.2 not too low as otherwise we might get seperated groups. This simulation, especially the second part, took extremely long (~1 hour)

In [ ]:
#ms = [1.1, 1.25, 1.5, 2]
ms = [1, 1.1, 1.25, 1.5, 1.75, 2, 2.25, 2.5]
runs = 500 #runs per m
groups = [1, 2, 3]
arr_runs = np.zeros((len(groups), len(ms), runs, 2))

print("agents = 20, n = 50, epsilon = 0.2, groups = 1,2,3, con_in = 1, con_out = 0.2")

for g in range(len(groups)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 20, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups[g], "con_in": 1, "con_out": 0.2})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[g, m, r, 0] = t
            arr_runs[g, m, r, 1] = output
            
polarized_prop = np.zeros((len(groups), len(ms)))
for g in range(len(groups)):
    for m in range(len(ms)):
        outputs = arr_runs[g, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[g, m] = polarized / runs
        
plt.figure()
for g_idx, g in enumerate(groups):
    plt.plot(ms, polarized_prop[g_idx, :], marker="o", label=f"g = {g}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("groups_over_m.pdf", bbox_inches="tight")
plt.show()

print("agents = 20, n = 10, epsilon = 0.01, groups = 1,2,3, con_in = 1, con_out = 0.2")
for g in range(len(groups)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 20, 0, 10, 1, 0.01, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups[g], "con_in": 1, "con_out": 0.2})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[g, m, r, 0] = t
            arr_runs[g, m, r, 1] = output
            
polarized_prop = np.zeros((len(groups), len(ms)))
for g in range(len(groups)):
    for m in range(len(ms)):
        outputs = arr_runs[g, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[g, m] = polarized / runs
        
plt.figure()
for g_idx, g in enumerate(groups):
    plt.plot(ms, polarized_prop[g_idx, :], marker="o", label=f"g = {g}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("groups_over_m_n10epsilon001.pdf", bbox_inches="tight")

plt.show()

In [ ]:
#ms = [1.1, 1.25, 1.5, 2]
ms = [1, 1.1, 1.25, 1.5, 1.75, 2, 2.25, 2.5]
runs = 100 #runs per m
groups = [1, 2, 3]
arr_runs = np.zeros((len(groups), len(ms), runs, 2))

print("agents = 200, n = 50, epsilon = 0.2, groups = 1,2,3, con_in = 1, con_out = 0.2")
total = len(groups) * len(ms) * runs
done = 0

for g in range(len(groups)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 200, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups[g], "con_in": 1, "con_out": 0.2})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[g, m, r, 0] = t
            arr_runs[g, m, r, 1] = output
            
            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()
            
polarized_prop = np.zeros((len(groups), len(ms)))
for g in range(len(groups)):
    for m in range(len(ms)):
        outputs = arr_runs[g, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[g, m] = polarized / runs

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

plt.figure()
for g_idx, g in enumerate(groups):
    plt.plot(ms, polarized_prop[g_idx, :], marker="o", label=f"g = {g}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("groups_over_m_200agents.pdf", bbox_inches="tight")
plt.show()

##########################################

print("agents = 200, n = 10, epsilon = 0.01, groups = 1,2,3, con_in = 1, con_out = 0.2")
total = len(groups) * len(ms) * runs
done = 0

for g in range(len(groups)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 200, 0, 10, 1, 0.01, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups[g], "con_in": 1, "con_out": 0.2})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[g, m, r, 0] = t
            arr_runs[g, m, r, 1] = output

            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()
            
polarized_prop = np.zeros((len(groups), len(ms)))
for g in range(len(groups)):
    for m in range(len(ms)):
        outputs = arr_runs[g, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[g, m] = polarized / runs

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()
        
plt.figure()
for g_idx, g in enumerate(groups):
    plt.plot(ms, polarized_prop[g_idx, :], marker="o", label=f"g = {g}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("groups_over_m_n10epsilon001_200agents.pdf", bbox_inches="tight")

plt.show()

This did not have as much influence as i expected for the n=50, epsilon = 0.2 case. For the n = 10, epsilon = 0.01 case (uncertain info) we see even less influence of having several groups. So for the proportion of polarized runs we do not see too much influence of introducing groups, which is suprising to be honest. I would expect that because the groups are barely connected that we'd see for each group e.g. 50% change of polarization, that then for two groups we'd see the chance that either of them polarizes : 1-0.5*0.5 = 0.75. Instead we see that it is all roughly equal to eachother. 


Maybe the low influence in the initial case is because coun_out is so low? Lets see the influence of con_out with con_in = 1, 20 agents, and 2 groups. 

In [ ]:
#ms = [1.1, 1.25, 1.5, 2]
ms = [1, 1.1, 1.25, 1.5, 1.75, 2, 2.25, 2.5]
runs = 500 #runs per m
groups = 2
con_out = [0, 0.01, 0.025, 0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 1]
arr_runs = np.zeros((len(con_out), len(ms), runs, 2))

total = len(con_out) * len(ms) * runs
done = 0

print("agents = 20, n = 50, epsilon = 0.2, groups = 2, con_in = 1, con_out = var")

for c in range(len(con_out)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 20, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups, "con_in": 1, "con_out": con_out[c]})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[c, m, r, 0] = t
            arr_runs[c, m, r, 1] = output
            
            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()
            
polarized_prop = np.zeros((len(con_out), len(ms)))
for c in range(len(con_out)):
    for m in range(len(ms)):
        outputs = arr_runs[c, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[c, m] = polarized / runs

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()
        
plt.figure()
for c_idx, c in enumerate(con_out):
    plt.plot(ms, polarized_prop[c_idx, :], marker="o", label=f"con_out = {c}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("pol_over_con_out.pdf", bbox_inches="tight")
plt.show()

############################33

total = len(con_out) * len(ms) * runs
done = 0
print("agents = 200, n = 50, epsilon = 0.2, groups = 2, con_in = 1, con_out = var")

for c in range(len(con_out)):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 200, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups, "con_in": 1, "con_out": con_out[c]})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[c, m, r, 0] = t
            arr_runs[c, m, r, 1] = output
            
            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()
            
polarized_prop = np.zeros((len(con_out), len(ms)))
for c in range(len(con_out)):
    for m in range(len(ms)):
        outputs = arr_runs[c, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[c, m] = polarized / runs

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()
        
plt.figure()
for c_idx, c in enumerate(con_out):
    plt.plot(ms, polarized_prop[c_idx, :], marker="o", label=f"con_out = {c}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("pol_over_con_out_200agents.pdf", bbox_inches="tight")
plt.show()

Interesting result for larger groups; we see that for con_out around 0.05-0.01 we see the biggest shift, meaning this number of expernal connections (2 groups of 100 means roughly 2-10 connected agents in a different group and 99 in your own group so roughly 2-10% of your connections are external) has a big effect.

Again not that much influence... Why is this so? maybe more influence is found in the percentage of true believers? i.e. is this the same for the percentage of true beliefs? Yes indeed again we see that, for both the clear and vague information case, we have that the number of groups has little effect on the proportion of true believers. This is somewhat expected as we expect that the average of true believers in one seperate group mimics that of the total population.

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 500 #runs per m
groups = [1, 2, 3]
arr_runs = np.zeros((len(groups), len(ms), runs, 2))

false_prop = np.zeros((len(groups), len(ms)))
true_prop = np.zeros((len(groups), len(ms)))

print("agents = 20, n = 50, epsilon = 0.2, groups = 1,2,3, con_in = 1, con_out = 0.2")

total = len(groups) * len(ms) * runs
done = 0

for g in range(len(groups)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            arr_credence, _, t, _ = simulation2(10001, 20, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups[g], "con_in": 1, "con_out": 0.2})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[g, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[g, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

plt.figure()
for g_idx, g in enumerate(groups):
    plt.plot(ms, true_prop[g_idx, :], marker="o", label=f"groups = {g}")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

# handles, labels = plt.gca().get_legend_handles_labels()
# order = [4, 3, 2, 1, 0]  # example new order
# plt.legend([handles[i] for i in order],
#            [labels[i] for i in order])
plt.legend()

#plt.savefig("beliefs_over_cons.pdf", bbox_inches="tight")
plt.show()


#########################################################

print("agents = 20, n = 10, epsilon = 0.01, groups = 1,2,3, con_in = 1, con_out = 0.2")
total = len(groups) * len(ms) * runs
done = 0

for g in range(len(groups)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            arr_credence, _, t, _ = simulation2(10001, 20, 0, 10, 1, 0.01, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups[g], "con_in": 1, "con_out": 0.2})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()


        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[g, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[g, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

plt.figure()
for g_idx, g in enumerate(groups):
    plt.plot(ms, true_prop[g_idx, :], marker="o", label=f"groups = {g}")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

# handles, labels = plt.gca().get_legend_handles_labels()
# order = [4, 3, 2, 1, 0]  # example new order
# plt.legend([handles[i] for i in order],
#            [labels[i] for i in order])
plt.legend()

#plt.savefig("beliefs_over_consn10eps001.pdf", bbox_inches="tight")
plt.show()

So interestingly, for uncertain data there is almost no difference between having several groups, i believe this has to do with the time it takes to convergence and a longer time allows everything to mellow out. Whereas certain data leads to quicker convergence but also different outcomes for different number of groups.

Again lets check if con_out does not have a great effect on this, and indeed it does not, just like in the proportion of polarized runs case.

In [ ]:
ms = [1.1, 1.5, 2, 2.5]
runs = 100 #runs per m
con_out = [0.1, 0.25, 0.5, 0.75, 1]
arr_runs = np.zeros((len(con_out), len(ms), runs, 2))

false_prop = np.zeros((len(con_out), len(ms)))
true_prop = np.zeros((len(con_out), len(ms)))
print("agents = 20, n = 50, epsilon = 0.2, groups = 2, con_in = 1, con_out = variable")
for c in range(len(con_out)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            arr_credence, _, t, _ = simulation2(10001, 20, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": 2, "con_in": 1, "con_out": con_out[c]})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[c, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[c, m] = true_beliefs / (false_beliefs + true_beliefs)

plt.figure()
for c_idx, c in enumerate(con_out):
    plt.plot(ms, true_prop[c_idx, :], marker="o", label=f"con_out = {c}")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("beliefs_over_m_over_cons2agents.pdf", bbox_inches="tight")
plt.show()


ms = [1.1, 1.5, 2, 2.5]
runs = 100 #runs per m
con_out = [0.1, 0.25, 0.5, 0.75, 1]
arr_runs = np.zeros((len(con_out), len(ms), runs, 2))

false_prop = np.zeros((len(con_out), len(ms)))
true_prop = np.zeros((len(con_out), len(ms)))
print("agents = 100, n = 50, epsilon = 0.2, groups = 2, con_in = 1, con_out = variable")

for c in range(len(con_out)):        
    for m in range(len(ms)):
        false_beliefs = 0
        true_beliefs = 0
        for r in range(runs):
            arr_credence, _, t, _ = simulation2(10001, 100, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": 2, "con_in": 1, "con_out": con_out[c]})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func

            false_beliefs += np.sum(arr_credence[t, :] < 0.5)
            true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        #print(false_beliefs, true_beliefs, false_beliefs + true_beliefs)
        false_prop[c, m] = false_beliefs / (false_beliefs + true_beliefs)
        true_prop[c, m] = true_beliefs / (false_beliefs + true_beliefs)

plt.figure()
for c_idx, c in enumerate(con_out):
    plt.plot(ms, true_prop[c_idx, :], marker="o", label=f"con_out = {c}")

plt.xlabel("m")
plt.ylabel("Percentage of True Beliefs")
plt.ylim(0, 1)

handles, labels = plt.gca().get_legend_handles_labels()
order = [4, 3, 2, 1, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("beliefs_over_m_over_cons_100agents.pdf", bbox_inches="tight")
plt.show()

A good sanity check is to check if the proportion of polarized runs in 2 groups is squared compared to the normal case, if we have two groups with con_out = 0. Note that we are comparing then two groups of 10 agents vs 1 group of 20 agents, so (2 groups) is not exactly 1 - (1 - (1 groups))^2, but it should be close. 

Note, we get P_pol(2 group) = 1 - (1 - P_pol(1 group))^2  as P_npol(2 groups) = P_npol(1 group)^2 when con_out = 0 and the total populations are equally large.

In [ ]:
#ms = [1.1, 1.25, 1.5, 2]
ms = [1, 1.1, 1.25, 1.5, 1.75, 2, 2.25, 2.5]
runs = 500 #runs per m
groups = 2
con_out = [0, 1, 2]
arr_runs = np.zeros((len(con_out), len(ms), runs, 2))

print("agents = 20, n = 50, epsilon = 0.2, groups = 2, con_in = 1, con_out = 0, 1")

total = 2 * len(ms) * runs
done = 0

for c in range(2):        
    for m in range(len(ms)):
        for r in range(runs):
            #O'Connor and Weatherall use a variety of parameters, their figure 2 uses: tics=?, a=6,10,20, i=0, n=50, con=1, epsilon=0.2, m=...
            _, _, t, output = simulation2(10001, 20, 0, 50, 1, 0.2, ms[m], output_text=False, disc_func=2, topology={"type": "groups", "num_groups": groups, "con_in": 1, "con_out": con_out[c]})
            #tics, a, i, n, con, epsilon, m, output_text, version, disc_func
            arr_runs[c, m, r, 0] = t
            arr_runs[c, m, r, 1] = output
            
            done += 1
            percent = 100 * done / total
            sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
            sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()
            
polarized_prop = np.zeros((len(con_out), len(ms)))
for c in range(2):
    for m in range(len(ms)):
        outputs = arr_runs[c, m, :, 1]
        polarized = np.sum(outputs == 3)
        polarized_prop[c, m] = polarized / runs

for m in range(len(ms)):
    polarized_prop[2,m] = 1 - ((1 - polarized_prop[1,m]) ** 2)
        
plt.figure()
for c_idx, c in enumerate(con_out):
    plt.plot(ms, polarized_prop[c_idx, :], marker="o", label=f"con_out = {c}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
#plt.savefig("pol_over_con_out.pdf", bbox_inches="tight")
plt.show()

This most interesting revelation here is that having group topology while still sparcely connected does not seem to lead to very differing results. One last thing I wish to check is fro which very low con_out, with high agent population, we do start seeing the above behaviour.

### This was the group topology, but there are many others, i will investigate the small world graph
This looks like a wheel but for larger populations, where closer agents are more connected than far away agents. This is shown to be a realistic way of modeling scientific communities (https://pmc.ncbi.nlm.nih.gov/articles/PMC14598/ Newman 2001, and https://www.pnas.org/doi/10.1073/pnas.0307545100 Newman 2004). For the watts-strogatz parameters to be used in our simulation, we can use table 1 from Newman 2001 to retrieve reasonable numbers. From the average number of collaborators we can set k = 8 (anywhere from 4 to 15 is realistic), and we can set p = 0.1 (which is a normal pick here to preserve small-world properties but not colapse into a random graph)

In [ ]:
graph, _, _ = generate_graph_smallworld(100, 0, 1, 10, 0.05) #a, i, tics, k, p
plot_graph(graph)

In [ ]:
arr = simulation2(10001, 100, 0, 10, 1, 0.01, 1.1, output_text=True, disc_func=2, topology={"type": "smallworld", "k": 8, "p": 0.1})
#tics, a, i, n, con, epsilon, m, output_text, version, disc_func
print("agents = 100, n = 10, epsilon = 0.01, m = 1.1, smallworld, k = 8, p = 0.1")
line_plot_credence(*arr[:3])

This looks quite similar to a combination of the low connectivity and the multiple group settings, lets look at the polarization results and the ratio of true beliefs.

First lets compare the polarization result to a random network of similar connectedness (if k = 8, with 100 agents, then we have c = 0.08). We expect to see higher polarization than 

In [ ]:
runs = 250
tics = 1001

ms = [1, 1.05, 1.1, 1.15, 1.25, 1.5, 1.75, 2]
c = 0.08
arr_runs_c008 = np.zeros((len(ms), runs, 2))

total = 3 * len(ms) * runs
done = 0
for m_idx, m_val in enumerate(ms):
    for r in range(runs):
        _, _, t, output = run_simulation(tics, 100, 0, 50, c, 0.2, m_val, False, 2, 2)
        arr_runs_c008[m_idx, r, 0] = t
        arr_runs_c008[m_idx, r, 1] = output

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


polarized_prop_c008 = np.zeros(len(ms))
for m_idx in range(len(ms)):
    outputs = arr_runs_c008[m_idx, :, 1]
    polarized_prop_c008[m_idx] = np.sum(outputs == 3) / runs


###########################

c = 1
arr_runs_c1 = np.zeros((len(ms), runs, 2))

for m_idx, m_val in enumerate(ms):
    for r in range(runs):
        _, _, t, output = run_simulation(tics, 100, 0, 50, c, 0.2, m_val, False, 2, 2)
        arr_runs_c1[m_idx, r, 0] = t
        arr_runs_c1[m_idx, r, 1] = output

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


polarized_prop_c1 = np.zeros(len(ms))
for m_idx in range(len(ms)):
    outputs = arr_runs_c1[m_idx, :, 1]
    polarized_prop_c1[m_idx] = np.sum(outputs == 3) / runs


################################

k = 8
p = 0.1
arr_runs_sw = np.zeros((len(ms), runs, 2))

#print("agents = 100, n = 50, epsilon = 0.2, small world, k = 8, p = 0.1")
for m_idx, m_val in enumerate(ms):
    for r in range(runs):
        _, _, t, output = simulation2(tics, 100, 0, 50, 1, 0.2, m_val, output_text=False, disc_func=2, topology={"type": "smallworld", "k": k, "p": p})
        arr_runs_sw[m_idx, r, 0] = t
        arr_runs_sw[m_idx, r, 1] = output

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


polarized_prop_sw = np.zeros(len(ms))
for m_idx in range(len(ms)):
    outputs = arr_runs_sw[m_idx, :, 1]
    polarized_prop_sw[m_idx] = np.sum(outputs == 3) / runs


plt.figure()

plt.plot(ms, polarized_prop_c008, marker="o", label="Random network, c = 0.08")
plt.plot(ms, polarized_prop_c1, marker="o", label="Random network, c = 1")
plt.plot(ms, polarized_prop_sw, marker="s", label=f"Small-world, k = {k}, p = {p}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)


handles, labels = plt.gca().get_legend_handles_labels()
order = [0, 2, 1]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("c008_vs_c1_vs_smallworld_k8_p01_100a_ep02_n50.pdf")
plt.show()

In [ ]:
runs = 100
tics = 1001

ms = [1.1, 1.25, 1.5, 1.75, 2, 2.5]
c = 0.08

false_prop = np.zeros((3, len(ms)))
true_prop = np.zeros((3, len(ms)))

total = 3 * len(ms) * runs
done = 0
for m, m_val in enumerate(ms):
    false_beliefs = 0
    true_beliefs = 0
    for r in range(runs):
        arr_credence, _, t, _ = run_simulation(tics, 100, 0, 50, c, 0.2, m_val, False, 2, 2)
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)
        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

    false_prop[0, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[0, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

###########################

c = 1


for m, m_val in enumerate(ms):
    false_beliefs = 0
    true_beliefs = 0
    for r in range(runs):
        arr_credence, _, t, _ = run_simulation(tics, 100, 0, 50, c, 0.2, m_val, False, 2, 2)
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)
        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

    false_prop[1, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[1, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


################################

k = 8
p = 0.1


#print("agents = 100, n = 50, epsilon = 0.2, small world, k = 8, p = 0.1")
for m, m_val in enumerate(ms):
    for r in range(runs):
        arr_credence, _, t, _ = simulation2(tics, 100, 0, 50, 1, 0.2, m_val, output_text=False, disc_func=2, topology={"type": "smallworld", "k": k, "p": p})
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()
        
    false_prop[2, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[2, m] = true_beliefs / (false_beliefs + true_beliefs)
    
sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


plt.figure()

plt.plot(ms, true_prop[0,:], marker="o", label="Random network, c = 0.08")
plt.plot(ms, true_prop[1,:], marker="o", label="Random network, c = 1")
plt.plot(ms, true_prop[2,:], marker="s", label=f"Small-world, k = {k}, p = {p}")

plt.xlabel("m")
plt.ylabel("Percentage of True Believers")
plt.ylim(0, 1)
plt.grid(True)


handles, labels = plt.gca().get_legend_handles_labels()
order = [1, 2, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("true_beliefs_c008_vs_c1_vs_smallworld_k8_p01_100a_ep02_n50.pdf")
plt.show()

This result is quite surprising as we see now that the amount of true believers of the small-world model mimics closely the fully connected network, especially for higher m. This suggests that there are benificial properties from the fully connected network that the small-world network seems to have, without requiring full connectedness. This also points towards the idea that the litereature, on simulating with fully connected networks, is not far off reality (which we do not achieve here but we get arguably closer to).

When we compare the polarization plot 2 above we see a different trend though; the small-world mimics the low connected network more than the fully connected one, but neither are really close, so clearly the small-world also has some properties that enables polarization.

Notable is also the percentage of true believers at very low m, as it looks like both randomly (c = 0.08 and c = 1) connected networks start at almost 1, whereas the small world starts somehwat lower, something that we, for n = 50 and epsilon = 0.2 (high information), have not seen before; we have only seen this in low information dense settings.


Let us redo the above plots for n = 10, eps = 0.01. Note, the below plot took ~10 hours to render but its outcome is very interesting... Need to investigate this further!!!!!

In [ ]:
runs = 250
tics = 50001

ms = [1, 1.015, 1.03, 1.05, 1.1, 1.15, 1.25, 1.5]
c = 0.08
arr_runs_c008 = np.zeros((len(ms), runs, 2))

total = 3 * len(ms) * runs
done = 0
for m_idx, m_val in enumerate(ms):
    for r in range(runs):
        _, _, t, output = run_simulation(tics, 100, 0, 10, c, 0.01, m_val, False, 2, 2)
        arr_runs_c008[m_idx, r, 0] = t
        arr_runs_c008[m_idx, r, 1] = output

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


polarized_prop_c008 = np.zeros(len(ms))
for m_idx in range(len(ms)):
    outputs = arr_runs_c008[m_idx, :, 1]
    polarized_prop_c008[m_idx] = np.sum(outputs == 3) / runs


###########################

c = 1
arr_runs_c1 = np.zeros((len(ms), runs, 2))

for m_idx, m_val in enumerate(ms):
    for r in range(runs):
        _, _, t, output = run_simulation(tics, 100, 0, 10, c, 0.01, m_val, False, 2, 2)
        arr_runs_c1[m_idx, r, 0] = t
        arr_runs_c1[m_idx, r, 1] = output

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


polarized_prop_c1 = np.zeros(len(ms))
for m_idx in range(len(ms)):
    outputs = arr_runs_c1[m_idx, :, 1]
    polarized_prop_c1[m_idx] = np.sum(outputs == 3) / runs


################################

k = 8
p = 0.1
arr_runs_sw = np.zeros((len(ms), runs, 2))

for m_idx, m_val in enumerate(ms):
    for r in range(runs):
        _, _, t, output = simulation2(tics, 100, 0, 10, 1, 0.01, m_val, output_text=False, disc_func=2, topology={"type": "smallworld", "k": k, "p": p})
        arr_runs_sw[m_idx, r, 0] = t
        arr_runs_sw[m_idx, r, 1] = output

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


polarized_prop_sw = np.zeros(len(ms))
for m_idx in range(len(ms)):
    outputs = arr_runs_sw[m_idx, :, 1]
    polarized_prop_sw[m_idx] = np.sum(outputs == 3) / runs


plt.figure()

plt.plot(ms, polarized_prop_c008, marker="o", label="Random network, c = 0.08")
plt.plot(ms, polarized_prop_c1, marker="o", label="Random network, c = 1")
plt.plot(ms, polarized_prop_sw, marker="s", label=f"Small-world, k = {k}, p = {p}")

plt.xlabel("m")
plt.ylabel("Proportion of Polarized Runs")
plt.ylim(0, 1)
plt.grid(True)


handles, labels = plt.gca().get_legend_handles_labels()
order = [0, 2, 1]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("c008_vs_c1_vs_smallworld_k8_p01_100a_ep001_n10v2.pdf")
plt.show()

In [ ]:
runs = 100
tics = 10001

ms = [1.1, 1.25, 1.5, 1.75, 2, 2.5]
c = 0.08

false_prop = np.zeros((3, len(ms)))
true_prop = np.zeros((3, len(ms)))

total = 3 * len(ms) * runs
done = 0
for m, m_val in enumerate(ms):
    false_beliefs = 0
    true_beliefs = 0
    for r in range(runs):
        arr_credence, _, t, _ = run_simulation(tics, 100, 0, 10, c, 0.01, m_val, False, 2, 2)
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)
        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

    false_prop[0, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[0, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

###########################

c = 1


for m, m_val in enumerate(ms):
    false_beliefs = 0
    true_beliefs = 0
    for r in range(runs):
        arr_credence, _, t, _ = run_simulation(tics, 100, 0, 10, c, 0.01, m_val, False, 2, 2)
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)
        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

    false_prop[1, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[1, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


################################

k = 8
p = 0.1


#print("agents = 100, n = 50, epsilon = 0.2, small world, k = 8, p = 0.1")
for m, m_val in enumerate(ms):
    for r in range(runs):
        arr_credence, _, t, _ = simulation2(tics, 100, 0, 10, 1, 0.01, m_val, output_text=False, disc_func=2, topology={"type": "smallworld", "k": k, "p": p})
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()
        
    false_prop[2, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[2, m] = true_beliefs / (false_beliefs + true_beliefs)
    
sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


plt.figure()

plt.plot(ms, true_prop[0,:], marker="o", label="Random network, c = 0.08")
plt.plot(ms, true_prop[1,:], marker="o", label="Random network, c = 1")
plt.plot(ms, true_prop[2,:], marker="s", label=f"Small-world, k = {k}, p = {p}")

plt.xlabel("m")
plt.ylabel("Percentage of True Believers")
plt.ylim(0, 1)
plt.grid(True)


handles, labels = plt.gca().get_legend_handles_labels()
order = [1, 2, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("true_beliefs_c008_vs_c1_vs_smallworld_k8_p01_100a_ep001_n10.pdf")
plt.show()

trial with 100.000 tics (so check if we get convergence in above one)

In [ ]:
runs = 5
tics = 100001

ms = [1.1, 1.25, 1.5, 1.75, 2, 2.5]
c = 0.08

false_prop = np.zeros((3, len(ms)))
true_prop = np.zeros((3, len(ms)))

total = 3 * len(ms) * runs
done = 0
for m, m_val in enumerate(ms):
    false_beliefs = 0
    true_beliefs = 0
    for r in range(runs):
        arr_credence, _, t, _ = run_simulation(tics, 100, 0, 10, c, 0.01, m_val, False, 2, 2)
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)
        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

    false_prop[0, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[0, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()

###########################

c = 1


for m, m_val in enumerate(ms):
    false_beliefs = 0
    true_beliefs = 0
    for r in range(runs):
        arr_credence, _, t, _ = run_simulation(tics, 100, 0, 10, c, 0.01, m_val, False, 2, 2)
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)
        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()

    false_prop[1, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[1, m] = true_beliefs / (false_beliefs + true_beliefs)

sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


################################

k = 8
p = 0.1


#print("agents = 100, n = 50, epsilon = 0.2, small world, k = 8, p = 0.1")
for m, m_val in enumerate(ms):
    for r in range(runs):
        arr_credence, _, t, _ = simulation2(tics, 100, 0, 10, 1, 0.01, m_val, output_text=False, disc_func=2, topology={"type": "smallworld", "k": k, "p": p})
        false_beliefs += np.sum(arr_credence[t, :] < 0.5)
        true_beliefs += np.sum(arr_credence[t, :] >= 0.5)

        done += 1
        percent = 100 * done / total
        sys.stdout.write(f"\rProgress: {percent:6.2f}% ({done}/{total})")
        sys.stdout.flush()
        
    false_prop[2, m] = false_beliefs / (false_beliefs + true_beliefs)
    true_prop[2, m] = true_beliefs / (false_beliefs + true_beliefs)
    
sys.stdout.write("\r" + " " * 50 + "\r")
sys.stdout.flush()


plt.figure()

plt.plot(ms, true_prop[0,:], marker="o", label="Random network, c = 0.08")
plt.plot(ms, true_prop[1,:], marker="o", label="Random network, c = 1")
plt.plot(ms, true_prop[2,:], marker="s", label=f"Small-world, k = {k}, p = {p}")

plt.xlabel("m")
plt.ylabel("Percentage of True Believers")
plt.ylim(0, 1)
plt.grid(True)


handles, labels = plt.gca().get_legend_handles_labels()
order = [1, 2, 0]  # example new order
plt.legend([handles[i] for i in order],
           [labels[i] for i in order])

#plt.savefig("true_beliefs_c008_vs_c1_vs_smallworld_k8_p01_100a_ep001_n10_runs5tics100000.pdf")
plt.show()

## After contact with Jonathan Weisber, I aim to look at the effect that the order of agents has on the outcome of a agent with begining credence 0.5, and after one tic

This section is exploratory and was written to investigate order-dependence in a targeted way. It is less polished than the main simulation functions and should be read as an exploratory extension rather than as part of the core model.

This code is very janky as it serves one purpose, it is altered from existing code, and i was mostly testing to see what for effects different things have. Right now it does this:
- Runs simulation style 2 (o'connor and weatherall) for one tic and outputs the exp results and distribution of where an agent with 0.5 credence ends up after that one tic
- Runs simulation style 1 (mine with saved credences that update all at once), using the same exp results and initial credences as sim style 2 did above.

Right now you need to run this all in order to make sure no errors occur and it works properly.

In [ ]:
mean_and_std_list = []

In [ ]:
#SIMULATION 2 Style (i.e O'Connor and Weatherall style of updating)

# per tic
# 1. everyone experiments; they set k but DONT update Bayesian
# 2. social updating is done order dependant:
#    for each agent (in increasing index order):
#            a) If the agent experimented, Bayes-update on own evidence first
#            b) Then Jeffrey-update on each neighbor's evidence (using neighbor's current credence).
# Stopping condition:
#   - stop if all credences > .99  (outcome 1)
#   - stop if all credences <= .5  (outcome 2)
#   - stop if d * m > 1 meaning polarized state
#   - else continue until tics cap (outcome 0)

tics = 2
a = 20
i = 0
n = 10
con = 1
epsilon = 0.01
m = 1.2
topology = None
disc_func =2
num_orders = 1000

if topology is None:
    topology = {"type": "random", "con": con}

graph, arr_credence_original, exp_results = build_graph(a, i, tics, topology)

def initial_credences2(arr_credence):
    n_agents = arr_credence.shape[1]
    arr_credence[0, :] = np.linspace(0, 1, n_agents)
    return arr_credence

arr_credence_original = initial_credences(arr_credence_original)

arr_credence_original[0,0] = 0.5 #agent 0 does not experiment

arr_credence = arr_credence_original


cred_begin = arr_credence[0, :]

outcome = 0
t_end = 0

t = 1
#start this tic from last tic credences where no-one has updated yet
arr_credence[t, :] = arr_credence[t - 1, :]

#experiments: ONLY draw k (store evidence). No Bayesian update here.
for agent in range(a+i):
    cred = arr_credence[t - 1, agent]
    if cred > 0.5:
        exp_results[t, agent] = np.random.binomial(n, 0.5 + epsilon)
    else:
        exp_results[t, agent] = -1


valid = exp_results[1, exp_results[1, :] != -1]
num_exp = len(valid)
sum_success = valid.sum() if num_exp > 0 else 0
avg_rate = sum_success / (num_exp * n) if num_exp > 0 else np.nan

#cred_end = np.zeros((num_orders, a+i))
end_vals_2 = np.zeros(num_orders)

for j in range(num_orders):
    order = np.random.permutation(a + i)
    arr_credence[t, :] = arr_credence[t - 1, :].copy()
    
    
    #social updating in order, so dependant on the order we do it
    for agent in order:
        #self Bayes update first (if experimented)
        if exp_results[t, agent] != -1:
            arr_credence[t, agent] = bayesian_update_self(exp_results[t, agent], arr_credence[t, agent], n, epsilon)
    
        # jeffrey updating, excluding self
        for neighbour in order:
            if neighbour == agent:
                continue
            if graph[agent, neighbour] == 1 and exp_results[t, neighbour] != -1:
                arr_credence[t, agent] = update_on_neighbours(arr_credence[t, agent], arr_credence[t, neighbour], exp_results[t, neighbour], n, epsilon, m, disc_func=disc_func)

    #cred_end[j,:] = arr_credence[1, :]
    end_vals_2[j] = arr_credence[1, 0]


In [ ]:
# fixed evidence realization at t=1
evidence = exp_results[1, :]
initial = arr_credence[0, :]

agents = np.arange(a + i)
experimented = evidence != -1
rates = np.full(a + i, np.nan)
rates[experimented] = evidence[experimented] / n

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.2, 1]})

# ---------------------------
# Left panel: evidence profile
# ---------------------------
ax = axes[0]

# bars for experimenters
ax.bar(
    agents[experimented],
    rates[experimented],
    width=0.8,
    alpha=0.7,
    label="Observed success rate"
)

# markers for non-experimenters
ax.scatter(
    agents[~experimented],
    np.zeros(np.sum(~experimented)),
    marker="x",
    s=60,
    label="Did not experiment"
)

# # highlight agent 0
# ax.axvline(0, linestyle="--", linewidth=1)
# ax.scatter([0], [0], s=80, zorder=5, label="Agent 0")

ax.set_xlabel("Agent")
ax.set_ylabel("Observed success rate")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Fixed evidence realization at t = 1")

# # overlay initial credences
# ax2 = ax.twinx()
# ax2.plot(agents, initial, marker="o", linewidth=1.5, alpha=0.8, label="Initial credence")
# ax2.set_ylabel("Initial credence")
# ax2.set_ylim(-0.02, 1.02)

# combine legends
handles1, labels1 = ax.get_legend_handles_labels()
ax.legend(handles1, labels1, loc="upper left", fontsize=9)

# add a summary text box
valid = evidence[experimented]
num_exp = len(valid)
sum_success = valid.sum() if num_exp > 0 else 0
avg_rate = sum_success / (num_exp * n) if num_exp > 0 else np.nan

summary = (
    f"Experimenters: {num_exp}/{a+i}\n"
    f"Mean observed rate: {avg_rate:.3f}\n"
    f"Agent 0 initial credence: {initial[0]:.2f}"
)
ax.text(
    0.98, 0.02, summary,
    transform=ax.transAxes,
    ha="right", va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

# ---------------------------
# Right panel: outcome distribution
# ---------------------------
ax = axes[1]

counts, bins = np.histogram(end_vals_2, bins=100)
probabilities = counts / counts.sum()

ax.bar(bins[:-1], probabilities, width=np.diff(bins), align='edge')
ax.set_xlabel("Final credence of agent 0")
ax.set_ylabel("Probability")
ax.set_title("Distribution over update orders")

# show central tendency
ax.axvline(np.mean(end_vals_2), linestyle="--", linewidth=1, label=f"Mean = {np.mean(end_vals_2):.3f}, Std = {np.std(end_vals_2):.3f}")
ax.legend()

plt.tight_layout()
#plt.savefig("order_runs100_e02_n50_10a_m2_numorders1000.pdf")
plt.show()

In [ ]:
# SIMULATION 1 Style (symmetric / synchronous updating)

# per tic
# 1. everyone experiments at the same time:
#    if credence > 0.5:
#         draw k and Bayes-update on own evidence
#    else:
#         no experiment, credence unchanged
# 2. social updating is synchronous:
#    take snapshot of post-experiment credences
#    each agent updates on neighbours using that fixed snapshot
#    store results in a separate working array
# 3. after all neighbour-updating is computed, write final values for tic t 

#commented to compare the two at same starting place
# Use the same graph builder as style 2
    # #graph, arr_credence, exp_results_1 = build_graph(a, i, tics, topology) 
    
    # arr_credence = arr_credence_original
    
    # def initial_credences2(arr_credence):
    #     n_agents = arr_credence.shape[1]
    #     arr_credence[0, :] = np.linspace(0, 1, n_agents)
    #     return arr_credence
    
    # arr_credence = initial_credences(arr_credence)

    # agent 0 does not experiment
    #arr_credence[0, 0] = 0.5
arr_credence = arr_credence_original

cred_begin = arr_credence[0, :]

outcome = 0
t_end = 0

t = 1

# start tic 1 from tic 0 credences
arr_credence[t, :] = arr_credence[t - 1, :]


# step 1: experiment + self-Bayes update
exp_results_1 = exp_results
for agent in range(a + i): #updating order does not matter here
    cred = arr_credence[t - 1, agent]

    if cred > 0.5: #commented to compare the two at same starting place
    #     k = np.random.binomial(n, 0.5 + epsilon)
    #     exp_results_1[t, agent] = k

        # Bayes update on own evidence, starting from previous credence
        arr_credence[t, agent] = bayesian_update_self(exp_results_1[t, agent], arr_credence[t - 1, agent], n, epsilon)
    else:
        exp_results_1[t, agent] = -1
        arr_credence[t, agent] = arr_credence[t - 1, agent]

# stats for fixed evidence realization
valid = exp_results_1[t, exp_results_1[t, :] != -1]
num_exp = len(valid)
sum_success = valid.sum() if num_exp > 0 else 0
avg_rate = sum_success / (num_exp * n) if num_exp > 0 else np.nan

# step 2: synchronous social updating

# Snapshot after self-updating, before social updating
post_exp_credence = arr_credence[t, :].copy()

end_vals_1 = np.zeros(num_orders)

for j in range(num_orders):
    post_social_credence = post_exp_credence.copy()
    order = np.random.permutation(a + i)
    
    for agent in order:
        current_cred = post_exp_credence[agent]
    
        for neighbour in order:
            if neighbour == agent:
                continue
    
            if graph[agent, neighbour] == 1 and exp_results_1[t, neighbour] != -1:
                current_cred = update_on_neighbours(current_cred, post_exp_credence[neighbour], exp_results_1[t, neighbour], n, epsilon, m,disc_func=disc_func)
    
        post_social_credence[agent] = current_cred
        
    end_vals_1[j] = post_social_credence[0]

In [ ]:
# fixed evidence realization at t=1
evidence = exp_results_1[1, :]
initial = arr_credence[0, :]

agents = np.arange(a + i)
experimented = evidence != -1
rates = np.full(a + i, np.nan)
rates[experimented] = evidence[experimented] / n

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.2, 1]})

# ---------------------------
# Left panel: evidence profile
# ---------------------------
ax = axes[0]

# bars for experimenters
ax.bar(
    agents[experimented],
    rates[experimented],
    width=0.8,
    alpha=0.7,
    label="Observed success rate"
)

# markers for non-experimenters
ax.scatter(
    agents[~experimented],
    np.zeros(np.sum(~experimented)),
    marker="x",
    s=60,
    label="Did not experiment"
)

# # highlight agent 0
# ax.axvline(0, linestyle="--", linewidth=1)
# ax.scatter([0], [0], s=80, zorder=5, label="Agent 0")

ax.set_xlabel("Agent")
ax.set_ylabel("Observed success rate")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Fixed evidence realization at t = 1")

# # overlay initial credences
# ax2 = ax.twinx()
# ax2.plot(agents, initial, marker="o", linewidth=1.5, alpha=0.8, label="Initial credence")
# ax2.set_ylabel("Initial credence")
# ax2.set_ylim(-0.02, 1.02)

# combine legends
handles1, labels1 = ax.get_legend_handles_labels()
ax.legend(handles1, labels1, loc="upper left", fontsize=9)

# add a summary text box
valid = evidence[experimented]
num_exp = len(valid)
sum_success = valid.sum() if num_exp > 0 else 0
avg_rate = sum_success / (num_exp * n) if num_exp > 0 else np.nan

summary = (
    f"Experimenters: {num_exp}/{a+i}\n"
    f"Mean observed rate: {avg_rate:.3f}\n"
    f"Agent 0 initial credence: {initial[0]:.2f}"
)
ax.text(
    0.98, 0.02, summary,
    transform=ax.transAxes,
    ha="right", va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

# ---------------------------
# Right panel: outcome distribution
# ---------------------------
ax = axes[1]

counts, bins = np.histogram(end_vals_1, bins=100)
probabilities = counts / counts.sum()

ax.bar(bins[:-1], probabilities, width=np.diff(bins), align='edge')
ax.set_xlabel("Final credence of agent 0")
ax.set_ylabel("Probability")
ax.set_title("Distribution over update orders")

# show central tendency
ax.axvline(np.mean(end_vals_1), linestyle="--", linewidth=1, label=f"Mean = {np.mean(end_vals_1):.3f}, Std = {np.std(end_vals_1):.3f}")
ax.legend()

plt.tight_layout()
#plt.savefig("order_runs100_e02_n50_10a_m2_numorders1000_sim1.pdf")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(end_vals_1, bins=100, density=True, alpha=0.5, color='blue', label='Synchronous updating style')
plt.hist(end_vals_2, bins=100, density=True, alpha=0.5, color='red', label="O'Connor and Weatherall style")

summary = (
    f"Experimenters: {num_exp}/{a+i}\n"
    f"Mean observed rate: {avg_rate:.3f}\n"
    f"Agent 0 initial credence: {initial[0]:.2f}"
)
plt.text(
    0.98, 0.6, summary,
    transform=plt.gca().transAxes,
    ha="right", va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

plt.xlabel("Final credence of agent 0")
plt.ylabel("Density")
plt.title("Distribution over update orders")
plt.legend()
plt.tight_layout()
#plt.savefig("order_runs100_e02_n50_10a_m2_numorders1000_sim12_combined.pdf")
plt.show()

In [ ]:
mean_1 = np.mean(end_vals_1)
std_1 = np.std(end_vals_1)

mean_2 = np.mean(end_vals_2)
std_2 = np.std(end_vals_2)

mean_and_std_list.append([mean_1, std_1, mean_2, std_2])


In [ ]:
mean_and_std_list = []

for variation in range(5): #increase this to 100 to see very clear behaviour, be aware, this will take a few minutes
    #SIMULATION 2 Style (i.e O'Connor and Weatherall style of updating)
    
    # per tic
    # 1. everyone experiments; they set k but DONT update Bayesian
    # 2. social updating is done order dependant:
    #    for each agent (in increasing index order):
    #            a) If the agent experimented, Bayes-update on own evidence first
    #            b) Then Jeffrey-update on each neighbor's evidence (using neighbor's current credence).
    # Stopping condition:
    #   - stop if all credences > .99  (outcome 1)
    #   - stop if all credences <= .5  (outcome 2)
    #   - stop if d * m > 1 meaning polarized state
    #   - else continue until tics cap (outcome 0)
    
    tics = 2
    a = 50
    i = 0
    n = 10
    con = 1
    epsilon = 0.01
    m = 1.2
    topology = None
    disc_func =2
    num_orders = 250
    
    if topology is None:
        topology = {"type": "random", "con": con}
    
    graph, arr_credence_original, exp_results = build_graph(a, i, tics, topology)
    
    def initial_credences2(arr_credence):
        n_agents = arr_credence.shape[1]
        arr_credence[0, :] = np.linspace(0, 1, n_agents)
        return arr_credence
    
    arr_credence_original = initial_credences(arr_credence_original)
    
    arr_credence_original[0,0] = 0.5 #agent 0 does not experiment
    
    arr_credence = arr_credence_original
    
    
    cred_begin = arr_credence[0, :]
    
    outcome = 0
    t_end = 0
    
    t = 1
    #start this tic from last tic credences where no-one has updated yet
    arr_credence[t, :] = arr_credence[t - 1, :]
    
    #experiments: ONLY draw k (store evidence). No Bayesian update here.
    for agent in range(a+i):
        cred = arr_credence[t - 1, agent]
        if cred > 0.5:
            exp_results[t, agent] = np.random.binomial(n, 0.5 + epsilon)
        else:
            exp_results[t, agent] = -1
    
    
    valid = exp_results[1, exp_results[1, :] != -1]
    num_exp = len(valid)
    sum_success = valid.sum() if num_exp > 0 else 0
    avg_rate = sum_success / (num_exp * n) if num_exp > 0 else np.nan
    
    #cred_end = np.zeros((num_orders, a+i))
    end_vals_2 = np.zeros(num_orders)
    
    for j in range(num_orders):
        order = np.random.permutation(a + i)
        arr_credence[t, :] = arr_credence[t - 1, :].copy()
        
        
        #social updating in order, so dependant on the order we do it
        for agent in order:
            #self Bayes update first (if experimented)
            if exp_results[t, agent] != -1:
                arr_credence[t, agent] = bayesian_update_self(exp_results[t, agent], arr_credence[t, agent], n, epsilon)
        
            # jeffrey updating, excluding self
            for neighbour in order:
                if neighbour == agent:
                    continue
                if graph[agent, neighbour] == 1 and exp_results[t, neighbour] != -1:
                    arr_credence[t, agent] = update_on_neighbours(arr_credence[t, agent], arr_credence[t, neighbour], exp_results[t, neighbour], n, epsilon, m, disc_func=disc_func)
    
        #cred_end[j,:] = arr_credence[1, :]
        end_vals_2[j] = arr_credence[1, 0]
    
    
    
    #sim 1
    # SIMULATION 1 Style (symmetric / synchronous updating)
    
    # per tic
    # 1. everyone experiments at the same time:
    #    if credence > 0.5:
    #         draw k and Bayes-update on own evidence
    #    else:
    #         no experiment, credence unchanged
    # 2. social updating is synchronous:
    #    take snapshot of post-experiment credences
    #    each agent updates on neighbours using that fixed snapshot
    #    store results in a separate working array
    # 3. after all neighbour-updating is computed, write final values for tic t 
    
    #commented to compare the two at same starting place
    # Use the same graph builder as style 2
        # #graph, arr_credence, exp_results_1 = build_graph(a, i, tics, topology) 
        
        # arr_credence = arr_credence_original
        
        # def initial_credences2(arr_credence):
        #     n_agents = arr_credence.shape[1]
        #     arr_credence[0, :] = np.linspace(0, 1, n_agents)
        #     return arr_credence
        
        # arr_credence = initial_credences(arr_credence)
    
        # agent 0 does not experiment
        #arr_credence[0, 0] = 0.5
    arr_credence = arr_credence_original
    
    cred_begin = arr_credence[0, :]
    
    outcome = 0
    t_end = 0
    
    t = 1
    
    # start tic 1 from tic 0 credences
    arr_credence[t, :] = arr_credence[t - 1, :]
    
    
    # step 1: experiment + self-Bayes update
    exp_results_1 = exp_results
    for agent in range(a + i): #updating order does not matter here
        cred = arr_credence[t - 1, agent]
    
        if cred > 0.5: #commented to compare the two at same starting place
        #     k = np.random.binomial(n, 0.5 + epsilon)
        #     exp_results_1[t, agent] = k
    
            # Bayes update on own evidence, starting from previous credence
            arr_credence[t, agent] = bayesian_update_self(exp_results_1[t, agent], arr_credence[t - 1, agent], n, epsilon)
        else:
            exp_results_1[t, agent] = -1
            arr_credence[t, agent] = arr_credence[t - 1, agent]
    
    # stats for fixed evidence realization
    valid = exp_results_1[t, exp_results_1[t, :] != -1]
    num_exp = len(valid)
    sum_success = valid.sum() if num_exp > 0 else 0
    avg_rate = sum_success / (num_exp * n) if num_exp > 0 else np.nan
    
    # step 2: synchronous social updating
    
    # Snapshot after self-updating, before social updating
    post_exp_credence = arr_credence[t, :].copy()
    
    end_vals_1 = np.zeros(num_orders)
    
    for j in range(num_orders):
        post_social_credence = post_exp_credence.copy()
        order = np.random.permutation(a + i)
        
        for agent in order:
            current_cred = post_exp_credence[agent]
        
            for neighbour in order:
                if neighbour == agent:
                    continue
        
                if graph[agent, neighbour] == 1 and exp_results_1[t, neighbour] != -1:
                    current_cred = update_on_neighbours(current_cred, post_exp_credence[neighbour], exp_results_1[t, neighbour], n, epsilon, m,disc_func=disc_func)
        
            post_social_credence[agent] = current_cred
            
        end_vals_1[j] = post_social_credence[0]
    
    
    mean_1 = np.mean(end_vals_1)
    std_1 = np.std(end_vals_1)
    
    mean_2 = np.mean(end_vals_2)
    std_2 = np.std(end_vals_2)
    
    mean_and_std_list.append([num_exp, avg_rate, mean_1, std_1, mean_2, std_2])

In [ ]:
data = np.array(mean_and_std_list)

avg_rate_vals = data[:, 1]

mean_1_vals = data[:, 2]
std_1_vals  = data[:, 3]

mean_2_vals = data[:, 4]
std_2_vals  = data[:, 5]

idx = np.argsort(avg_rate_vals)

avg_rate_vals = avg_rate_vals[idx]

mean_1_vals = mean_1_vals[idx]
std_1_vals  = std_1_vals[idx]

mean_2_vals = mean_2_vals[idx]
std_2_vals  = std_2_vals[idx]

plt.figure(figsize=(7,5))

# Style 1 (blue)
plt.errorbar(
    avg_rate_vals,
    mean_1_vals,
    yerr=std_1_vals,
    fmt='o',
    color='blue',
    alpha=0.7,
    capsize=3,
    label='Synchronous'
)

# Style 2 (red)
plt.errorbar(
    avg_rate_vals,
    mean_2_vals,
    yerr=std_2_vals,
    fmt='o',
    color='red',
    alpha=0.7,
    capsize=3,
    label='Sequential (O&W)'
)

plt.xlabel("Mean observed rate")
plt.ylabel("Final credence of agent 0")
plt.title("Mean outcome with order-sensitivity (std)")

plt.legend()
plt.tight_layout()
#plt.savefig("mean_and_std_runs100_e02_n50_50a_m12_numorders250.pdf")
plt.show()